In [ ]:
from fractions import Fraction
from typing import List, Tuple, Optional

def to_frac(x):
    """Преобразует число (int/float/str/Fraction) в Fraction."""
    return x if isinstance(x, Fraction) else Fraction(x)

def simplex_canonical(
    c: List,
    A: List[List],
    b: List,
    maximize: bool = True
) -> Optional[Tuple[List[Fraction], Fraction]]:
    """
    Решает задачу ЛП в канонической форме:
        если maximize=True:  максимизировать c^T x
        если maximize=False: минимизировать c^T x
    при условиях A x = b, x >= 0.
    Предполагается, что начальный базис допустим (например, единичная подматрица).
    """
    # Сохраняем оригинальное направление
    original_c = [to_frac(x) for x in c]
    if not maximize:
        c = [-to_frac(x) for x in c]  # минимизация → максимизация -c^T x
    else:
        c = original_c

    # Далее — всё как раньше, с использованием c (возможно, инвертированного)
    A = [[to_frac(x) for x in row] for row in A]
    b = [to_frac(x) for x in b]

    m = len(A)
    n = len(c)

    assert len(b) == m, "Длина b должна совпадать с числом строк A"
    assert all(len(row) == n for row in A), "Несогласованность размеров A и c"
    assert all(bi >= 0 for bi in b), "Требуется b >= 0"

    basis = list(range(n - m, n))

    # (Проверка базиса — по желанию, можно оставить)

    iteration = 0
    while True:
        iteration += 1
        print(f"\n--- Итерация {iteration} ---")

        # Построение таблицы — без изменений
        table = [[Fraction(0)] * (n + 1) for _ in range(m + 1)]

        for i in range(m):
            for j in range(n):
                table[i + 1][j] = A[i][j]
            table[i + 1][n] = b[i]

        for j in range(n):
            table[0][j] = -c[j]

        if iteration > 1:
        current_z = Fraction(0)
        for i in range(m):
            cb = c[basis[i]]
            current_z += cb * b[i]
            for j in range(n + 1):
                table[0][j] += cb * table[i + 1][j]

        # Вывод таблицы с метками строк
        header = "\t" + "\t".join([f"x{j}" for j in range(n)] + ["RHS"])
        print(header)
        print(f"z\t" + "\t".join(str(val) for val in table[0]))
        for i in range(m):
            basic_var = basis[i]
            print(f"x{basic_var}\t" + "\t".join(str(val) for val in table[i + 1]))

        # Важно: значение целевой функции в терминах исходной задачи
        actual_z = current_z if maximize else -current_z
        print(f"Текущее значение целевой функции: {actual_z}")

        # Условие оптимальности — без изменений (всегда максимизация внутренне)
        if all(table[0][j] >= 0 for j in range(n)):
            print("Оптимальное решение найдено.")
            solution = [Fraction(0)] * n
            for i in range(m):
                solution[basis[i]] = b[i]
            optimal_value = current_z if maximize else -current_z
            return solution, optimal_value

        # Выбор ведущего столбца и строки — без изменений
        entering = next(j for j in range(n) if table[0][j] < 0)
        print(f"Входит переменная x{entering}")

        if all(table[i + 1][entering] <= 0 for i in range(m)):
            print("Целевая функция не ограничена." + 
                  (" сверху" if maximize else " снизу"))
            return None

        ratios = [
            (table[i + 1][n] / table[i + 1][entering], i)
            for i in range(m)
            if table[i + 1][entering] > 0
        ]
        _, leaving_row = min(ratios, key=lambda x: x[0])
        leaving_var = basis[leaving_row]
        print(f"Покидает базис переменная x{leaving_var}")

        basis[leaving_row] = entering

        pivot = table[leaving_row + 1][entering]
        for j in range(n + 1):
            table[leaving_row + 1][j] /= pivot

        for i in range(m + 1):
            if i == leaving_row + 1:
                continue
            factor = table[i][entering]
            for j in range(n + 1):
                table[i][j] -= factor * table[leaving_row + 1][j]

        for i in range(m):
            for j in range(n):
                A[i][j] = table[i + 1][j]
            b[i] = table[i + 1][n]

In [ ]:
if __name__ == "__main__":

    c = [3, -1, 3,0,-2,0,0,0,-1,0]
    A = [
        [1,1,0,-3,0,2,1,0,0,0],
        [2,0,-3,1,1,0,0,1,0,0],
        [3,-1,3,0,-2,0,0,0,-1,1]
    ]
    b = [5,4,3]
    sol, val = simplex_canonical(c, A, b, maximize = False)
    print("\nРешение:", [str(x) for x in sol])
    print("Оптимальное значение:", val)


--- Итерация 1 ---
	x0	x1	x2	x3	x4	x5	x6	x7	x8	x9	RHS
z	5	-1	0	1	-1	0	0	1	-1	0	4
x7	1	1	0	-3	0	2	1	0	0	0	5
x8	2	0	-3	1	1	0	0	1	0	0	4
x9	3	-1	3	0	-2	0	0	0	-1	1	3
Текущее значение целевой функции: -4
Входит переменная x1
Покидает базис переменная x7

--- Итерация 2 ---
	x0	x1	x2	x3	x4	x5	x6	x7	x8	x9	RHS
z	6	0	0	-2	-1	2	1	1	-1	0	9
x1	1	1	0	-3	0	2	1	0	0	0	5
x8	2	0	-3	1	1	0	0	1	0	0	4
x9	4	0	3	-3	-2	2	1	0	-1	1	8
Текущее значение целевой функции: -9
Входит переменная x3
Покидает базис переменная x8

--- Итерация 3 ---
	x0	x1	x2	x3	x4	x5	x6	x7	x8	x9	RHS
z	10	0	-6	0	1	2	1	3	-1	0	17
x1	7	1	-9	0	3	2	1	3	0	0	17
x3	2	0	-3	1	1	0	0	1	0	0	4
x9	10	0	-6	0	1	2	1	3	-1	1	20
Текущее значение целевой функции: -17
Входит переменная x2
Целевая функция не ограничена. снизу


TypeError: cannot unpack non-iterable NoneType object

In [13]:
from scipy.optimize import linprog

c = [2, -3, 0, 0, 1, 2]
A_ub = [
    [1, 1, 0, -3, 0, 2],
    [2, 0, -3, 1, 1, 0],
    [-3, 1, -3, 0, 2, 0]
]
b_ub = [5, 4, -3]

res = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=[(0, None)]*6, method='highs')

if res.success:
    print("Оптимальное решение:")
    for i, val in enumerate(res.x, 1):
        print(f"x{i} = {val}")
    print(f"f_min = {res.fun}")
else:
    print("Задача несовместна или не ограничена:", res.message)

Задача несовместна или не ограничена: The problem is unbounded. (HiGHS Status 10: model_status is Unbounded; primal_status is Feasible)
